In [ ]:
BEAM_WIDTH = 2**26 + 15_506_660
WORLD_SIZE = 2
START_PUZZLE_ID = 991
PUZZLE_COUNT = 9
DEPTH_LIMIT = 72
RUN_TIMEOUT_SEC = 0
GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_BRANCH = "main"
SOURCE_ARCHIVE_DATASET = "cayley-beam-solver-source-multigpu"
SOURCE_ARCHIVE_NAME = "beam_solver_source.tar.gz"
ENABLE_DEBUG = True
ENABLE_DEPTH_LOGS = True
ENABLE_DEBUG_LOGS = False
DEBUG_STREAM_TIMING = False
DEBUG_INFERENCE_TRACE = False
DEBUG_PATH_TRACE = False
DEBUG_FINAL_VALIDATE = False
DEBUG_FINAL_EXCHANGE_TRACE = False
DEBUG_FINAL_HISTOGRAM_TRACE = False
DEBUG_STREAM4_HISTOGRAM_TRACE = False
DEBUG_DEPTH_FLOW_TRACE = False
DEBUG_PIPELINE_STATS = False
DEPTH_LOG_EVERY = 1
PUZZLE_LOG_EVERY = 1
HISTORY_MODE = "static_hybrid"
HISTORY_SLOT_COUNT = 2
HISTORY_WORKERS = 1
HISTORY_RAM_BYTES = 28 * 1024**3
HISTORY_DISK_BYTES = 59 * 1024**3
HISTORY_DISK_PATH = "/tmp/beam_history_arena"
SOLVED_NEIGHBORHOOD_RADIUS = 4
SOLVED_NEIGHBORHOOD_MAX_ENTRIES = 0
STREAM2_SUFFIX_RADIUS = 0
STREAM2_SUFFIX_BACKEND = "composed_permutations"
STREAM2_SUFFIX_MAX_COUNT = 0
STOP_ON_FAILURE = False
LIVE_LOG_RANKS = [0]  # "all", "none", or a list like [0, 1]
RUN_STREAM_BENCHMARK = False

RUNTIME_CONFIG_MODE = "manual"
SHARD_BUFFER_COUNT = 2
STREAM4_BATCH_ALIGNMENT = 1024
SHARD_CAPACITY_SCALE_PPM = 1125000
STREAM4_ACTIVE_SORT_SLOTS = 4
GLOBAL_SPILL_CAPACITY = 0
STREAM5_RECV_CAPACITY_SCALE_PPM = 1000000
GPU_HEADROOM_BYTES = 256 * 1024**2
B_MICRO = 2048
STREAM1_CONCURRENCY = 4
STREAM3_RING_SLOTS = 4
SHARD_COUNT = 64
STREAM4_BATCH_CANDIDATES = 196608
STREAM4_TRIGGER_CANDIDATES = 393216

SWEEP_CONFIGS = [
    {
        "name": "p991_999_beam82m_sh64_s4_b2048_c4_r4_b196_t393",
        "B_MICRO": 2048,
        "STREAM1_CONCURRENCY": 4,
        "STREAM3_RING_SLOTS": 4,
        "SHARD_COUNT": 64,
        "STREAM4_BATCH_CANDIDATES": 196608,
        "STREAM4_TRIGGER_CANDIDATES": 393216,
        "STREAM4_ACTIVE_SORT_SLOTS": 4,
    },
]

In [ ]:
from pathlib import Path
import shutil

MOVE_COUNT = 24
HISTORY_ENTRY_BYTES = 16
CANDIDATE_META_BYTES = 32
HISTORY_WRITE_CHUNK_ENTRIES = 1 << 20
KAGGLE_DISK_RESERVE_BYTES = globals().get('KAGGLE_DISK_RESERVE_BYTES', 8 * 1024**3)


def round_up(value: int, alignment: int) -> int:
    return ((value + alignment - 1) // alignment) * alignment


def notebook_cleanup_path(path: Path):
    if not path.exists():
        return
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()


def estimate_history_budget_entries(depth_limit: int, k1_radius: int, k2_radius: int, beam_entries: int):
    suffix_radius = k1_radius + k2_radius
    effective_depth = max(depth_limit - suffix_radius, 0)
    frontier_bound = 1
    states_before_target = 0
    for depth in range(effective_depth):
        if frontier_bound < beam_entries:
            frontier_bound = min(beam_entries, frontier_bound * MOVE_COUNT)
        if frontier_bound >= beam_entries:
            full_depths = effective_depth - depth
            return {
                'effective_depth': effective_depth,
                'target_beam_depth': depth,
                'states_before_target_beam': states_before_target,
                'required_entries': states_before_target + full_depths * beam_entries,
            }
        states_before_target += frontier_bound
    return {
        'effective_depth': effective_depth,
        'target_beam_depth': effective_depth,
        'states_before_target_beam': states_before_target,
        'required_entries': states_before_target,
    }


def preflight_config(cfg: dict):
    shard_count = int(cfg.get('SHARD_COUNT', SHARD_COUNT))
    alignment = int(cfg.get('STREAM4_BATCH_ALIGNMENT', STREAM4_BATCH_ALIGNMENT))
    b_micro = int(cfg.get('B_MICRO', B_MICRO))
    ring_slots = int(cfg.get('STREAM3_RING_SLOTS', STREAM3_RING_SLOTS))
    concurrency = int(cfg.get('STREAM1_CONCURRENCY', STREAM1_CONCURRENCY))
    s4_batch = int(cfg.get('STREAM4_BATCH_CANDIDATES', STREAM4_BATCH_CANDIDATES))
    s4_trigger = int(cfg.get('STREAM4_TRIGGER_CANDIDATES', STREAM4_TRIGGER_CANDIDATES))
    scale_ppm = int(cfg.get('SHARD_CAPACITY_SCALE_PPM', SHARD_CAPACITY_SCALE_PPM))

    stream3_batch = b_micro * MOVE_COUNT * ring_slots
    beam_alignment = WORLD_SIZE * shard_count * alignment
    global_beam_effective = round_up(BEAM_WIDTH, beam_alignment)
    local_beam_width = global_beam_effective // WORLD_SIZE
    logical_shard_size = (local_beam_width + shard_count - 1) // shard_count
    shard_capacity = round_up((logical_shard_size * scale_ppm + 999999) // 1000000, alignment)

    pinned_per_rank = HISTORY_SLOT_COUNT * local_beam_width * CANDIDATE_META_BYTES
    staging_per_rank = HISTORY_SLOT_COUNT * min(local_beam_width, HISTORY_WRITE_CHUNK_ENTRIES) * HISTORY_ENTRY_BYTES
    ram_budget_per_rank = HISTORY_RAM_BYTES // WORLD_SIZE if HISTORY_RAM_BYTES else 0
    disk_budget_per_rank = HISTORY_DISK_BYTES // WORLD_SIZE if HISTORY_DISK_BYTES else 0
    ram_arena_per_rank = max(0, (ram_budget_per_rank - pinned_per_rank - staging_per_rank) // HISTORY_ENTRY_BYTES * HISTORY_ENTRY_BYTES)
    disk_arena_per_rank = disk_budget_per_rank // HISTORY_ENTRY_BYTES * HISTORY_ENTRY_BYTES
    estimate = estimate_history_budget_entries(
        DEPTH_LIMIT,
        SOLVED_NEIGHBORHOOD_RADIUS,
        STREAM2_SUFFIX_RADIUS,
        local_beam_width,
    )
    required_history_per_rank = estimate['required_entries'] * HISTORY_ENTRY_BYTES

    errors = []
    if concurrency < 1 or concurrency > ring_slots:
        errors.append(f'STREAM1_CONCURRENCY={concurrency} must be in [1, STREAM3_RING_SLOTS={ring_slots}]')
    if s4_batch < stream3_batch:
        errors.append(f'STREAM4_BATCH_CANDIDATES={s4_batch} must be >= STREAM3_BATCH_CANDIDATES={stream3_batch}')
    if s4_trigger < s4_batch:
        errors.append(f'STREAM4_TRIGGER_CANDIDATES={s4_trigger} must be >= STREAM4_BATCH_CANDIDATES={s4_batch}')
    if ram_budget_per_rank <= pinned_per_rank + staging_per_rank:
        errors.append(
            f'HISTORY_RAM_BYTES per rank too small: ram_per_rank={ram_budget_per_rank} '
            f'pinned={pinned_per_rank} staging={staging_per_rank}'
        )
    if required_history_per_rank > ram_arena_per_rank + disk_arena_per_rank:
        errors.append(
            f'static history budget too small per rank: required={required_history_per_rank} '
            f'ram_arena={ram_arena_per_rank} disk_arena={disk_arena_per_rank}'
        )

    history_disk_root = Path(HISTORY_DISK_PATH) if HISTORY_DISK_PATH else Path('/tmp/beam_history_arena')
    disk_probe = history_disk_root.parent if history_disk_root.parent != Path('') else Path('.')
    disk_free = shutil.disk_usage(disk_probe).free
    static_disk_total = disk_arena_per_rank * WORLD_SIZE
    projected_peak = static_disk_total + KAGGLE_DISK_RESERVE_BYTES
    if static_disk_total and projected_peak > disk_free:
        errors.append(
            f'disk preflight failed: static_disk_total={static_disk_total} reserve={KAGGLE_DISK_RESERVE_BYTES} '
            f'projected_peak={projected_peak} free_before_compile={disk_free} path={disk_probe}'
        )

    return {
        'name': cfg.get('name', 'default'),
        'stream3_batch': stream3_batch,
        'global_beam_effective': global_beam_effective,
        'local_beam_width': local_beam_width,
        'logical_shard_size': logical_shard_size,
        'shard_capacity': shard_capacity,
        'required_history_per_rank': required_history_per_rank,
        'ram_arena_per_rank': ram_arena_per_rank,
        'disk_arena_per_rank': disk_arena_per_rank,
        'static_disk_total': static_disk_total,
        'disk_free_before_compile': disk_free,
        'disk_reserve': KAGGLE_DISK_RESERVE_BYTES,
        'history_estimate': estimate,
        'errors': errors,
    }


history_root = Path(HISTORY_DISK_PATH) if HISTORY_DISK_PATH else None
if history_root is not None and history_root.exists():
    print(f'preflight_cleanup_history_path={history_root}')
    notebook_cleanup_path(history_root)

tmp_root = Path('/tmp')
if tmp_root.exists():
    for stale_history_path in tmp_root.glob('beam_history_*'):
        print(f'preflight_cleanup_history_path={stale_history_path}')
        notebook_cleanup_path(stale_history_path)

preflight_rows = [preflight_config(cfg) for cfg in SWEEP_CONFIGS]
for row in preflight_rows:
    print(
        'preflight_config'
        f' name={row["name"]}'
        f' stream3_batch={row["stream3_batch"]}'
        f' local_beam_width={row["local_beam_width"]}'
        f' shard_capacity={row["shard_capacity"]}'
        f' required_history_per_rank={row["required_history_per_rank"]}'
        f' ram_arena_per_rank={row["ram_arena_per_rank"]}'
        f' disk_arena_per_rank={row["disk_arena_per_rank"]}'
        f' static_disk_total={row["static_disk_total"]}'
        f' disk_free_before_compile={row["disk_free_before_compile"]}'
        f' disk_reserve={row["disk_reserve"]}'
        f' errors={len(row["errors"])}'
    )
    for error in row['errors']:
        print(f'preflight_error name={row["name"]} message={error}')

preflight_errors = [error for row in preflight_rows for error in row['errors']]
if preflight_errors:
    raise RuntimeError('Kaggle preflight failed before compilation; fix config first')

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

WORK_DIR = Path('/kaggle/working')
TMP_DIR = Path('/tmp')
REPO_DIR = TMP_DIR / 'beam_solver'
CUTLASS_DIR = TMP_DIR / 'cutlass'
BUILD_DIR = TMP_DIR / 'beam_build'

def run_checked(cmd, cwd=None, env=None):
    print('+ ' + ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

SOURCE_ARCHIVE_PATH = Path('/kaggle/input') / SOURCE_ARCHIVE_DATASET / SOURCE_ARCHIVE_NAME

for transient_dir in (REPO_DIR, BUILD_DIR):
    if transient_dir.exists():
        shutil.rmtree(transient_dir)
if SOURCE_ARCHIVE_PATH.exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    run_checked(['tar', '-xzf', SOURCE_ARCHIVE_PATH, '-C', REPO_DIR])
else:
    run_checked(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPO_URL, REPO_DIR])

if not (CUTLASS_DIR / 'include').exists():
    if CUTLASS_DIR.exists():
        shutil.rmtree(CUTLASS_DIR)
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/cutlass.git', CUTLASS_DIR])

depth_logs = 'ON' if ENABLE_DEPTH_LOGS else 'OFF'
debug_logs = 'ON' if ENABLE_DEBUG_LOGS else 'OFF'
debug_master = 'ON' if ENABLE_DEBUG else 'OFF'
debug_stream_timing = 'ON' if DEBUG_STREAM_TIMING else 'OFF'
debug_inference_trace = 'ON' if DEBUG_INFERENCE_TRACE else 'OFF'
debug_path_trace = 'ON' if DEBUG_PATH_TRACE else 'OFF'
debug_final_validate = 'ON' if DEBUG_FINAL_VALIDATE else 'OFF'
debug_final_exchange_trace = 'ON' if DEBUG_FINAL_EXCHANGE_TRACE else 'OFF'
debug_final_histogram_trace = 'ON' if DEBUG_FINAL_HISTOGRAM_TRACE else 'OFF'
debug_stream4_histogram_trace = 'ON' if DEBUG_STREAM4_HISTOGRAM_TRACE else 'OFF'
debug_depth_flow_trace = 'ON' if DEBUG_DEPTH_FLOW_TRACE else 'OFF'
run_checked([
    'cmake', '-S', REPO_DIR, '-B', BUILD_DIR, '-GNinja',
    '-DCMAKE_BUILD_TYPE=Release',
    f'-DCUTLASS_DIR={CUTLASS_DIR}',
    f'-DBEAM_ENABLE_DEBUG={debug_master}',
    f'-DBEAM_ENABLE_DEPTH_LOGS={depth_logs}',
    f'-DBEAM_ENABLE_DEBUG_LOGS={debug_logs}',
    f'-DBEAM_DEBUG_STREAM_TIMING={debug_stream_timing}',
    f'-DBEAM_DEBUG_INFERENCE_TRACE={debug_inference_trace}',
    f'-DBEAM_DEBUG_PATH_TRACE={debug_path_trace}',
    f'-DBEAM_DEBUG_FINAL_VALIDATE={debug_final_validate}',
    f'-DBEAM_DEBUG_FINAL_EXCHANGE_TRACE={debug_final_exchange_trace}',
    f'-DBEAM_DEBUG_FINAL_HISTOGRAM_TRACE={debug_final_histogram_trace}',
    f'-DBEAM_DEBUG_STREAM4_HISTOGRAM_TRACE={debug_stream4_histogram_trace}',
    f'-DBEAM_DEBUG_DEPTH_FLOW_TRACE={debug_depth_flow_trace}',
])
run_checked(['cmake', '--build', BUILD_DIR, '--target', 'production_runner', 'stream_benchmark', '-j', '2'])
bench_weight_dir = REPO_DIR / 'build-docker' / 'stream1_weights'
if not bench_weight_dir.exists():
    bench_weight_dir.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(REPO_DIR / 'stream1_weights', bench_weight_dir, target_is_directory=True)


In [ ]:
import re
import sys
import threading
import time
import shutil
import subprocess
import pandas as pd

KNOWN_SOLUTION_PATHS = {}

LOG_DIR = WORK_DIR / 'run_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = WORK_DIR / 'beam_run_results.csv'
SUBMISSION_CSV = WORK_DIR / 'submission.csv'
STREAM_BENCH_REPORT = WORK_DIR / 'stream_benchmark.md'
STREAM_BENCH_STDOUT = WORK_DIR / 'stream_benchmark_stdout.log'
SOLVED_RE = re.compile(r'puzzle_solved=(\d+) puzzle_id=(\d+) seconds=([0-9.eE+-]+) solution_length=(-?\d+) solution=(.*)$')


def round_up(value: int, alignment: int) -> int:
    return ((value + alignment - 1) // alignment) * alignment


def cfg_get(cfg: dict, key: str, default):
    return cfg[key] if key in cfg else default


def config_derived(cfg: dict) -> dict:
    shard_count = int(cfg_get(cfg, 'SHARD_COUNT', SHARD_COUNT if 'SHARD_COUNT' in globals() else 64))
    alignment = int(cfg_get(cfg, 'STREAM4_BATCH_ALIGNMENT', STREAM4_BATCH_ALIGNMENT))
    scale_ppm = int(cfg_get(cfg, 'SHARD_CAPACITY_SCALE_PPM', SHARD_CAPACITY_SCALE_PPM))
    b_micro = int(cfg_get(cfg, 'B_MICRO', 8192))
    stream3_ring_slots = int(cfg_get(cfg, 'STREAM3_RING_SLOTS', 1))
    stream3_batch = b_micro * 24 * stream3_ring_slots
    beam_alignment = WORLD_SIZE * shard_count * alignment
    global_beam_effective = round_up(BEAM_WIDTH, beam_alignment)
    local_beam_width = global_beam_effective // WORLD_SIZE
    logical_shard_size = (local_beam_width + shard_count - 1) // shard_count
    shard_capacity_raw = (logical_shard_size * scale_ppm + 999999) // 1000000
    shard_capacity = round_up(shard_capacity_raw, alignment)
    return {
        'b_micro': b_micro,
        'stream3_ring_slots': stream3_ring_slots,
        'stream3_batch_candidates': stream3_batch,
        'shard_count': shard_count,
        'beam_width_alignment': beam_alignment,
        'global_beam_width_effective': global_beam_effective,
        'local_beam_width': local_beam_width,
        'logical_shard_size': logical_shard_size,
        'shard_capacity_candidates': shard_capacity,
    }


def live_log_enabled(rank: int) -> bool:
    if LIVE_LOG_RANKS == 'all':
        return True
    if LIVE_LOG_RANKS == 'none':
        return False
    return rank in set(LIVE_LOG_RANKS)


def stream_rank_log(rank: int, proc, log_path: Path):
    with log_path.open('w', encoding='utf-8') as log:
        assert proc.stdout is not None
        for raw_line in proc.stdout:
            log.write(raw_line)
            log.flush()
            if live_log_enabled(rank):
                print(f'rank={rank} {raw_line}', end='')
                sys.stdout.flush()


def run_stream_benchmark_once():
    if not RUN_STREAM_BENCHMARK:
        return
    env = os.environ.copy()
    env['BEAM_STREAM_BENCH_REPORT'] = str(STREAM_BENCH_REPORT)
    env['BEAM_STREAM_MICRO_ONLY'] = '1'
    cmd = [str(BUILD_DIR / 'stream_benchmark'), str(START_PUZZLE_ID)]
    print('+ ' + ' '.join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    STREAM_BENCH_STDOUT.write_text(proc.stdout, encoding='utf-8')
    print(proc.stdout)
    print(f'stream_benchmark_return_code={proc.returncode}')
    print(f'stream_benchmark_report={STREAM_BENCH_REPORT}')



def cleanup_history_path(path: Path):
    if not path.exists():
        return
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()


def run_puzzle(puzzle_id: int, puzzle_index: int, cfg: dict, cfg_index: int):
    derived = config_derived(cfg)
    config_name = cfg['name']
    cfg_log_dir = LOG_DIR / config_name
    cfg_log_dir.mkdir(parents=True, exist_ok=True)
    history_path = TMP_DIR / f'beam_history_{config_name}_p{puzzle_id}'
    cleanup_history_path(history_path)

    env = os.environ.copy()
    env['BEAM_HISTORY_MODE'] = HISTORY_MODE
    env['BEAM_HISTORY_SLOT_COUNT'] = str(HISTORY_SLOT_COUNT)
    env['BEAM_HISTORY_WORKERS'] = str(HISTORY_WORKERS)
    env['BEAM_HISTORY_RAM_BYTES'] = str(HISTORY_RAM_BYTES)
    env['BEAM_HISTORY_DISK_BYTES'] = str(HISTORY_DISK_BYTES)
    env['BEAM_HISTORY_DISK_PATH'] = str(history_path)
    env['BEAM_SOLVED_NEIGHBORHOOD_RADIUS'] = str(SOLVED_NEIGHBORHOOD_RADIUS)
    env['BEAM_SOLVED_NEIGHBORHOOD_MAX_ENTRIES'] = str(SOLVED_NEIGHBORHOOD_MAX_ENTRIES)
    env['BEAM_STREAM2_SUFFIX_RADIUS'] = str(STREAM2_SUFFIX_RADIUS)
    env['BEAM_STREAM2_SUFFIX_BACKEND'] = STREAM2_SUFFIX_BACKEND
    env['BEAM_STREAM2_SUFFIX_MAX_COUNT'] = str(STREAM2_SUFFIX_MAX_COUNT)
    env['BEAM_DEPTH_LOG_EVERY'] = str(DEPTH_LOG_EVERY)
    if DEBUG_PIPELINE_STATS:
        env['BEAM_DEBUG_PIPELINE_STATS'] = '1'
    env['BEAM_WEIGHT_DIR'] = str(REPO_DIR / 'stream1_weights')
    env['BEAM_RUNTIME_CONFIG_MODE'] = RUNTIME_CONFIG_MODE
    env['BEAM_B_MICRO'] = str(derived['b_micro'])
    env['BEAM_STREAM1_CONCURRENCY'] = str(cfg_get(cfg, 'STREAM1_CONCURRENCY', STREAM1_CONCURRENCY))
    env['BEAM_SHARD_BUFFER_COUNT'] = str(SHARD_BUFFER_COUNT)
    env['BEAM_STREAM3_RING_SLOTS'] = str(derived['stream3_ring_slots'])
    env['BEAM_SHARD_COUNT'] = str(derived['shard_count'])
    env['BEAM_STREAM4_BATCH_CANDIDATES'] = str(cfg_get(cfg, 'STREAM4_BATCH_CANDIDATES', STREAM4_BATCH_CANDIDATES))
    env['BEAM_STREAM4_TRIGGER_CANDIDATES'] = str(cfg_get(cfg, 'STREAM4_TRIGGER_CANDIDATES', STREAM4_TRIGGER_CANDIDATES))
    env['BEAM_SHARD_CAPACITY_CANDIDATES'] = str(derived['shard_capacity_candidates'])
    env['BEAM_STREAM4_ACTIVE_SORT_SLOTS'] = str(cfg_get(cfg, 'STREAM4_ACTIVE_SORT_SLOTS', STREAM4_ACTIVE_SORT_SLOTS))
    env['BEAM_GLOBAL_SPILL_CAPACITY'] = str(GLOBAL_SPILL_CAPACITY)
    env['BEAM_STREAM5_RECV_CAPACITY_SCALE_PPM'] = str(STREAM5_RECV_CAPACITY_SCALE_PPM)
    env['BEAM_GPU_HEADROOM_BYTES'] = str(GPU_HEADROOM_BYTES)
    base_cmd = [str(BUILD_DIR / 'production_runner'), str(puzzle_id), str(DEPTH_LIMIT), str(BEAM_WIDTH)]
    print(
        'config_start'
        f' index={cfg_index}'
        f' name={config_name}'
        f' beam={BEAM_WIDTH}'
        f' depth_limit={DEPTH_LIMIT}'
        f' b_micro={derived["b_micro"]}'
        f' stream1_concurrency={env["BEAM_STREAM1_CONCURRENCY"]}'
        f' stream3_batch={derived["stream3_batch_candidates"]}'
        f' shard_count={derived["shard_count"]}'
        f' shard_capacity={derived["shard_capacity_candidates"]}'
        f' stream4_batch={env["BEAM_STREAM4_BATCH_CANDIDATES"]}'
        f' stream4_trigger={env["BEAM_STREAM4_TRIGGER_CANDIDATES"]}'
    )
    launch_id = f'{config_name}_p{puzzle_id}_{int(time.time())}'
    nccl_id_file = TMP_DIR / f'beam_solver_nccl_{launch_id}.bin'
    if nccl_id_file.exists():
        nccl_id_file.unlink()
    procs = []
    log_paths = []
    log_threads = []
    started = time.perf_counter()
    for rank in range(WORLD_SIZE):
        rank_env = env.copy()
        rank_env['WORLD_SIZE'] = str(WORLD_SIZE)
        rank_env['RANK'] = str(rank)
        rank_env['LOCAL_RANK'] = str(rank)
        rank_env['BEAM_NCCL_ID_FILE'] = str(nccl_id_file)
        log_path = cfg_log_dir / f'puzzle_{puzzle_id}_rank{rank}.log'
        proc = subprocess.Popen(base_cmd, cwd=REPO_DIR, env=rank_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        thread = threading.Thread(target=stream_rank_log, args=(rank, proc, log_path), daemon=True)
        thread.start()
        procs.append(proc)
        log_paths.append(log_path)
        log_threads.append(thread)
    timed_out = False
    deadline = started + RUN_TIMEOUT_SEC if RUN_TIMEOUT_SEC is not None and RUN_TIMEOUT_SEC > 0 else None
    while True:
        if all(proc.poll() is not None for proc in procs):
            break
        if deadline is not None and time.perf_counter() >= deadline:
            timed_out = True
            print(f'auto_stop_timeout_sec={RUN_TIMEOUT_SEC} config={config_name} puzzle_id={puzzle_id}')
            for proc in procs:
                if proc.poll() is None:
                    proc.terminate()
            grace_deadline = time.perf_counter() + 10.0
            while time.perf_counter() < grace_deadline and any(proc.poll() is None for proc in procs):
                time.sleep(0.2)
            for proc in procs:
                if proc.poll() is None:
                    proc.kill()
            break
        time.sleep(1.0)
    codes = [proc.wait() for proc in procs]
    for thread in log_threads:
        thread.join()
    elapsed = time.perf_counter() - started
    parsed = None
    for rank, log_path in enumerate(log_paths):
        with log_path.open('r', encoding='utf-8', errors='replace') as log:
            for raw_line in log:
                line = raw_line.rstrip(chr(10))
                match = SOLVED_RE.search(line)
                if match:
                    parsed = match
                    if not live_log_enabled(rank):
                        print(f'rank={rank} {line}')
                elif (not live_log_enabled(rank)) and (ENABLE_DEPTH_LOGS or line.startswith('track_solution_')):
                    print(f'rank={rank} {line}')
    failed_codes = [code for code in codes if code != 0]
    joined_logs = ';'.join(str(path) for path in log_paths)
    base_result = {
        'config': config_name,
        'config_index': cfg_index,
        'puzzle_id': puzzle_id,
        'beam_width': BEAM_WIDTH,
        'depth_limit': DEPTH_LIMIT,
        'b_micro': derived['b_micro'],
        'stream1_concurrency': int(env['BEAM_STREAM1_CONCURRENCY']),
        'stream3_ring_slots': derived['stream3_ring_slots'],
        'stream3_batch_candidates': derived['stream3_batch_candidates'],
        'shard_count': derived['shard_count'],
        'logical_shard_size': derived['logical_shard_size'],
        'shard_capacity_candidates': derived['shard_capacity_candidates'],
        'stream4_batch_candidates': int(env['BEAM_STREAM4_BATCH_CANDIDATES']),
        'stream4_trigger_candidates': int(env['BEAM_STREAM4_TRIGGER_CANDIDATES']),
        'stream4_active_sort_slots': int(env['BEAM_STREAM4_ACTIVE_SORT_SLOTS']),
        'global_beam_width_effective': derived['global_beam_width_effective'],
        'local_beam_width': derived['local_beam_width'],
        'seconds': elapsed,
        'log_path': joined_logs,
    }
    cleanup_history_path(history_path)
    if timed_out:
        return {**base_result, 'solved': 0, 'length': None, 'solution': '', 'return_code': -200}
    if failed_codes:
        result = {**base_result, 'solved': 0, 'length': None, 'solution': '', 'return_code': max(failed_codes)}
        if STOP_ON_FAILURE:
            raise RuntimeError(f'production_runner failed: config={config_name} puzzle_id={puzzle_id} return_codes={codes} log_paths={joined_logs}')
        return result
    if parsed is None:
        return {**base_result, 'solved': 0, 'length': None, 'solution': '', 'return_code': max(codes)}
    solved = int(parsed.group(1))
    return {
        **base_result,
        'puzzle_id': int(parsed.group(2)),
        'solved': solved,
        'seconds': float(parsed.group(3)),
        'length': int(parsed.group(4)) if solved else None,
        'solution': parsed.group(5) if solved else '',
        'return_code': max(codes),
    }


run_stream_benchmark_once()

results = []
for cfg_index, cfg in enumerate(SWEEP_CONFIGS, start=1):
    for puzzle_index, puzzle_id in enumerate(range(START_PUZZLE_ID, START_PUZZLE_ID + PUZZLE_COUNT), start=1):
        result = run_puzzle(puzzle_id, puzzle_index, cfg, cfg_index)
        results.append(result)
        if PUZZLE_LOG_EVERY and (puzzle_index % PUZZLE_LOG_EVERY == 0):
            print(
                f'config_progress={cfg_index}/{len(SWEEP_CONFIGS)} config={cfg["name"]} '
                f'puzzle_progress={puzzle_index}/{PUZZLE_COUNT} puzzle_id={puzzle_id} '
                f'solved={result["solved"]} seconds={result["seconds"]:.6f} length={result["length"]} '
                f'return_code={result["return_code"]}'
            )
        df = pd.DataFrame(results)
        df.to_csv(RESULTS_CSV, index=False)
        solved_df = df[df['solved'] == 1][['puzzle_id', 'solution']].rename(columns={'puzzle_id': 'initial_state_id', 'solution': 'path'})
        solved_df.to_csv(SUBMISSION_CSV, index=False)

pd.DataFrame(results)

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame(results)
solved = df[df['solved'] == 1].copy()
lengths = [int(x) for x in solved['length'].dropna().tolist()]
hist_path = WORK_DIR / 'solution_length_histogram.png'
if lengths:
    min_len = min(lengths)
    max_len = max(lengths)
    avg_len = sum(lengths) / len(lengths)
    mode_len = Counter(lengths).most_common(1)[0][0]
    plt.figure(figsize=(10, 5))
    plt.hist(lengths, bins=range(min_len, max_len + 2), edgecolor='black')
    plt.xlabel('solution_length')
    plt.ylabel('solved_puzzle_count')
    plt.title('Solved puzzle solution lengths')
    plt.tight_layout()
    plt.savefig(hist_path, dpi=160)
    print(f'solved_count={len(lengths)} total_count={PUZZLE_COUNT}')
    print(f'min_solution_length={min_len}')
    print(f'max_solution_length={max_len}')
    print(f'avg_solution_length={avg_len:.6f}')
    print(f'mode_solution_length={mode_len}')
    print(f'histogram_png={hist_path}')
else:
    print(f'solved_count=0 total_count={PUZZLE_COUNT}')
    print(f'histogram_png_not_created={hist_path}')

solved
